# Spain work–residence networks by MITMA recurrence layer
## Stage 0 — monthly data preparation and six-layer network construction, v2.0

This notebook retains every recurrence band supplied by the MITMA recurrent
district matrix instead of aggregating several bands into a single Hybrid or
Onsite network.

The official layers are:

| Recurrence band | Midpoint used for allocation | Interpretation |
|---|---:|---|
| 1–2 | 1.5 | very low-frequency / potentially irregular |
| 3–4 | 3.5 | low workplace recurrence |
| 5–7 | 6.0 | intermediate workplace recurrence |
| 8–10 | 9.0 | high workplace recurrence |
| 11–13 | 12.0 | very high workplace recurrence |
| 14 | 14.0 | recurrence on all fourteen natural days |

The recurrence variable refers to a fourteen-natural-day observation period
and therefore includes weekends. The layers are ordered by workplace
recurrence. They should not be described as directly observed remote-working
days or as individual worker types.

For each month, the notebook:

1. selects a complete Monday–Friday workweek near the middle of the month;
2. constructs mean weekday home-to-work OD flow;
3. reads the same month's recurrent district matrix;
4. allocates each OD's weekday work flow across all six recurrence bands in
   proportion to `people × recurrence midpoint`;
5. writes a wide six-layer edge table and a long layer table;
6. calculates resident-side and employment-side Hybrid-intensity indicators
   for every district.

The analytic Hybrid-intensity score is:

\[
H_b = 1 - \frac{m_b}{14},
\]

where \(m_b\) is the midpoint of recurrence band \(b\). Higher values indicate
lower recurrence at the same workplace location and therefore greater
workplace-location flexibility. This is an analytic ordering of the MITMA
layers, not a direct count of home-working days.

The original aggregated columns are retained only for backward compatibility.
All subsequent notebooks use the six official recurrence layers.

## 1. Install dependencies

In [ ]:
%pip install -q pandas pyarrow duckdb requests tqdm holidays geopandas pyogrio shapely

## 2. Configuration

In [ ]:
import os
from pathlib import Path

# ============================================================
# PROJECT AND TEMPORAL COVERAGE
# ============================================================

# Public-release path configuration.
# Set HYBRID_WORK_PROJECT_ROOT to an absolute project directory, or run the
# notebook from the project root. For example:
# PROJECT_ROOT = Path("/path/to/hybrid-work-network-project")
PROJECT_ROOT = Path(
    os.environ.get("HYBRID_WORK_PROJECT_ROOT", Path.cwd())
).expanduser().resolve()
START_MONTH = "2022-01"
END_MONTH = "2024-12"

# None means all months in the range.
MONTHS_TO_PROCESS = None
MAX_MONTHS_PER_RUN = None

RUN_PIPELINE = True
STOP_ON_ERROR = False

# ============================================================
# RESUME AND REPROCESSING
# ============================================================

SKIP_COMPLETED_MONTHS = True
REUSE_PARTIAL_OUTPUTS = True
REPROCESS_IF_CONFIG_CHANGED = True
FORCE_REPROCESS_MONTHS = set()

KEEP_RAW_DAILY = False
KEEP_RAW_RECURRENT = False
CLEAN_DAILY_SUMMARIES_AFTER_SUCCESS = True
KEEP_WORKWEEK_OD = True
KEEP_RECURRENT_PARQUET = True

# ============================================================
# TYPICAL WORKWEEK SELECTION
# ============================================================

TARGET_DAY_OF_MONTH = 15
EXCLUDE_NATIONAL_HOLIDAYS = True
ALLOW_HOLIDAY_WEEK_FALLBACK = False
EXPECTED_WEEKDAYS = 5

# ============================================================
# DAILY WORK-FLOW IDENTIFICATION
# ============================================================

USE_WORKING_AGE_FILTER = False
WORKING_AGE_VALUES = {"25-44", "45-64"}
REQUIRE_STUDY_FIELD = True
REQUIRE_EXPLICIT_NON_STUDY = True

ACTIVE_DAY_WORK_TRIP_THRESHOLD = 0.0
MIN_MEAN_WEEKDAY_WORK_TRIPS = 0.0

# ============================================================
# OPTIONAL DISTANCE FIELD
# ============================================================

REFERENCE_DIR = PROJECT_ROOT / "03_reference"
DISTRICT_CENTROID_SHP = REFERENCE_DIR / "zonificacion_distritos_centroides.shp"
ADD_DISTANCE_IF_AVAILABLE = True
DISTANCE_CRS = "EPSG:3035"
MIN_CENTROID_NODE_MATCH_SHARE = 0.90

# ============================================================
# OFFICIAL CATALOGUE
# ============================================================

RSS_URLS = [
    "https://movilidad-opendata.mitma.es/RSS.xml",
    "https://opendata-movilidad.mitma.es/RSS.xml",
]

# ============================================================
# MITMA RECURRENCE LAYERS
# ============================================================

RECURRENCE_BAND_ORDER = [
    "1-2",
    "3-4",
    "5-7",
    "8-10",
    "11-13",
    "14",
]

RECURRENCE_MIDPOINT = {
    "1-2": 1.5,
    "3-4": 3.5,
    "5-7": 6.0,
    "8-10": 9.0,
    "11-13": 12.0,
    "14": 14.0,
}

RECURRENCE_LAYER_LABEL = {
    "1-2": "1–2 recurrent days",
    "3-4": "3–4 recurrent days",
    "5-7": "5–7 recurrent days",
    "8-10": "8–10 recurrent days",
    "11-13": "11–13 recurrent days",
    "14": "14 recurrent days",
}

LAYER_WEIGHT_COLUMNS = {
    "1-2": "recurrence_1_2_weight",
    "3-4": "recurrence_3_4_weight",
    "5-7": "recurrence_5_7_weight",
    "8-10": "recurrence_8_10_weight",
    "11-13": "recurrence_11_13_weight",
    "14": "recurrence_14_weight",
}

# Analytic flexibility score. It is not a direct number of remote-work days.
HYBRID_INTENSITY_SCORE = {
    band: 1.0 - midpoint / 14.0
    for band, midpoint in RECURRENCE_MIDPOINT.items()
}

# Ordinal alternative used in robustness analyses.
HYBRID_RANK_SCORE = {
    band: rank / (len(RECURRENCE_BAND_ORDER) - 1)
    for rank, band in enumerate(reversed(RECURRENCE_BAND_ORDER))
}

# Legacy aggregates retained for comparison with older outputs.
LOW_FREQUENCY_BANDS = {"1-2"}
HYBRID_BANDS = {"3-4", "5-7"}
ONSITE_STANDARD_BANDS = {"8-10"}
ONSITE_VERY_HIGH_BANDS = {"11-13", "14"}
ONSITE_BROAD_BANDS = ONSITE_STANDARD_BANDS | ONSITE_VERY_HIGH_BANDS

# ============================================================
# STORAGE STRUCTURE
# ============================================================

TEMP_DIR = PROJECT_ROOT / "00_temp"
CATALOG_DIR = PROJECT_ROOT / "00_catalog"
DAILY_PARTIAL_DIR = PROJECT_ROOT / "01_intermediate" / "daily_workweek"
WORKWEEK_OD_DIR = PROJECT_ROOT / "01_intermediate" / "workweek_od"
RECURRENT_DIR = PROJECT_ROOT / "01_intermediate" / "recurrent"

LAYER_NETWORK_DIR = PROJECT_ROOT / "02_monthly_network_layers"
LAYER_LONG_DIR = LAYER_NETWORK_DIR / "long"
NODE_INTENSITY_DIR = LAYER_NETWORK_DIR / "node_intensity"

# Alias retained because several inherited utility cells use NETWORK_DIR.
NETWORK_DIR = LAYER_NETWORK_DIR

QC_DIR = PROJECT_ROOT / "04_qc" / "recurrence_layer_pipeline"
MONTH_STATUS_DIR = QC_DIR / "month_status"
RUN_LOG_DIR = QC_DIR / "run_logs"

for folder in [
    PROJECT_ROOT,
    TEMP_DIR,
    CATALOG_DIR,
    DAILY_PARTIAL_DIR,
    WORKWEEK_OD_DIR,
    RECURRENT_DIR,
    LAYER_NETWORK_DIR,
    LAYER_LONG_DIR,
    NODE_INTENSITY_DIR,
    REFERENCE_DIR,
    QC_DIR,
    MONTH_STATUS_DIR,
    RUN_LOG_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Coverage:", START_MONTH, "to", END_MONTH)
print("Wide layer networks:", LAYER_NETWORK_DIR)
print("Long layer networks:", LAYER_LONG_DIR)
print("Node intensity outputs:", NODE_INTENSITY_DIR)

## 3. Imports, run logging, and utility functions

In [ ]:
import gzip
import hashlib
import html
import json
import os
import platform
import re
import shutil
import time
import traceback
import unicodedata
import xml.etree.ElementTree as ET

from datetime import datetime
from urllib.parse import urljoin, urlparse, unquote

import duckdb
import holidays
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests

from IPython.display import display
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)

RUN_STARTED_AT = datetime.now()
RUN_ID = RUN_STARTED_AT.strftime("%Y%m%d_%H%M%S")
RUN_PERF_START = time.perf_counter()
RUN_STAGE_RECORDS = []


def normalise_token(value):
    if value is None:
        return ""
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = re.sub(r"[\s/\\\-]+", "_", value)
    value = re.sub(r"[^a-z0-9_<>+=.]", "", value)
    return re.sub(r"_+", "_", value).strip("_")


def canonical_district_id(value):
    if pd.isna(value):
        return None
    value = unicodedata.normalize("NFKC", str(value).strip())
    value = re.sub(r"\.0$", "", value)
    value = re.sub(r"\s+", "", value)
    return value.upper() if value else None


def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / 1024**2 if path.exists() else np.nan


def directory_size_mb(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0.0
    return sum(p.stat().st_size for p in folder.rglob("*") if p.is_file()) / 1024**2


def atomic_write_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8-sig")
    temp.replace(path)


def safe_read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def upsert_csv(records, path, keys):
    path = Path(path)
    incoming = pd.DataFrame(records if isinstance(records, list) else [records])
    if incoming.empty:
        return
    if path.exists():
        existing = pd.read_csv(path, dtype=str)
        combined = pd.concat([existing, incoming.astype(str)], ignore_index=True)
    else:
        combined = incoming.astype(str)
    combined = combined.drop_duplicates(subset=keys, keep="last")
    atomic_write_csv(combined, path)


def validate_parquet(path, required_columns, min_rows=1):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        parquet_file = pq.ParquetFile(path)
        available = set(parquet_file.schema.names)
        required = set(required_columns)
        return required.issubset(available) and parquet_file.metadata.num_rows >= min_rows
    except Exception:
        return False


def make_session():
    retry = Retry(
        total=6,
        connect=6,
        read=6,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET", "HEAD"),
    )
    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "Spain-Hybrid-Work-Research-Pipeline/1.0"
        )
    })
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.mount("http://", HTTPAdapter(max_retries=retry))
    return session


SESSION = make_session()


def xml_local_name(tag):
    return tag.split("}")[-1].lower()


def extract_file_temporal_metadata(filename):
    filename = str(filename)
    daily_match = re.search(r"(?<!\d)(20\d{6})(?!\d)", filename)
    if daily_match:
        file_date = pd.to_datetime(daily_match.group(1), format="%Y%m%d", errors="coerce")
        file_month = file_date.strftime("%Y-%m") if not pd.isna(file_date) else None
        return file_date, file_month

    monthly_match = re.search(r"(?<!\d)(20\d{4})(?!\d)", filename)
    if monthly_match:
        month_date = pd.to_datetime(monthly_match.group(1), format="%Y%m", errors="coerce")
        file_month = month_date.strftime("%Y-%m") if not pd.isna(month_date) else None
        return pd.NaT, file_month

    return pd.NaT, None


def month_range(start_month, end_month):
    start = pd.Period(start_month, freq="M")
    end = pd.Period(end_month, freq="M")
    return [str(period) for period in pd.period_range(start, end, freq="M")]


CONFIG_FOR_SIGNATURE = {
    "start_month": START_MONTH,
    "end_month": END_MONTH,
    "target_day_of_month": TARGET_DAY_OF_MONTH,
    "exclude_national_holidays": EXCLUDE_NATIONAL_HOLIDAYS,
    "allow_holiday_week_fallback": ALLOW_HOLIDAY_WEEK_FALLBACK,
    "use_working_age_filter": USE_WORKING_AGE_FILTER,
    "working_age_values": sorted(WORKING_AGE_VALUES),
    "require_study_field": REQUIRE_STUDY_FIELD,
    "require_explicit_non_study": REQUIRE_EXPLICIT_NON_STUDY,
    "minimum_mean_weekday_work_trips": MIN_MEAN_WEEKDAY_WORK_TRIPS,
    "recurrence_band_order": RECURRENCE_BAND_ORDER,
    "recurrence_midpoint": RECURRENCE_MIDPOINT,
    "layer_weight_columns": LAYER_WEIGHT_COLUMNS,
    "hybrid_intensity_score": HYBRID_INTENSITY_SCORE,
    "allocation_method": "people_times_midpoint_share",
}
CONFIG_SIGNATURE = hashlib.sha256(
    json.dumps(CONFIG_FOR_SIGNATURE, sort_keys=True).encode("utf-8")
).hexdigest()[:16]

# Record only portable run metadata. Hardware, disk, and local-path details
# are intentionally excluded from the public workflow.
system_info = {
    "run_id": RUN_ID,
    "run_started_at": RUN_STARTED_AT.isoformat(timespec="seconds"),
    "python": platform.python_version(),
    "config_signature": CONFIG_SIGNATURE,
}

atomic_write_csv(pd.DataFrame([system_info]), RUN_LOG_DIR / f"run_environment_{RUN_ID}.csv")
print("Run ID:", RUN_ID)
print("Configuration signature:", CONFIG_SIGNATURE)
display(pd.DataFrame([system_info]))

## 4. Retrieve and classify the official MITMA file catalogue

In [ ]:
def parse_catalog_xml(content, source_url):
    root = ET.fromstring(content)
    records = []

    for element in root.iter():
        tag = xml_local_name(element.tag)
        if tag not in {"item", "entry", "contents"}:
            continue

        record = {}
        for child in element.iter():
            key = xml_local_name(child.tag)
            text = (child.text or "").strip()
            if key in {"title", "key", "name"} and text:
                record.setdefault("title", text)
            if key in {"link", "url", "location"}:
                candidate = child.attrib.get("href") or text
                if candidate:
                    record.setdefault("link", candidate)
            if key in {"pubdate", "published", "updated", "lastmodified"} and text:
                record.setdefault("published", text)
            if key == "key" and text:
                record["key"] = text

        if record.get("key") and not record.get("link"):
            record["link"] = urljoin(source_url, record["key"])
        if record.get("link"):
            records.append(record)

    if not records:
        text = content.decode("utf-8", errors="ignore")
        urls = re.findall(r"https?://[^<\"'\s]+", text)
        records = [{"link": html.unescape(url)} for url in urls]

    rows = []
    for record in records:
        link = html.unescape(str(record.get("link", "")).strip())
        if not link:
            continue
        if not link.lower().startswith(("http://", "https://")):
            link = urljoin(source_url, link)

        filename = Path(unquote(urlparse(link).path)).name
        if not filename:
            filename = Path(str(record.get("title", ""))).name
        if not filename:
            continue

        file_date, file_month = extract_file_temporal_metadata(filename)
        rows.append({
            "filename": filename,
            "link": link,
            "title": record.get("title"),
            "published": record.get("published"),
            "file_date": file_date,
            "file_month": file_month,
        })

    return pd.DataFrame(rows).drop_duplicates(subset=["link"]).reset_index(drop=True)


def load_official_catalog():
    errors = []
    for url in RSS_URLS:
        try:
            print("Trying:", url)
            response = SESSION.get(url, timeout=180)
            response.raise_for_status()
            catalog = parse_catalog_xml(response.content, url)
            if catalog.empty:
                raise ValueError("No file records were parsed.")
            return catalog, url
        except Exception as error:
            errors.append(f"{url}: {error}")
    raise RuntimeError("Could not load official catalogue:\n" + "\n".join(errors))


def classify_catalog_file(filename):
    name = normalise_token(filename)

    basic_district_patterns = [
        r"^20\d{6}_viajes_distritos\.csv\.gz$",
        r"^20\d{6}_trips_districts\.csv\.gz$",
        r"^20\d{6}_trips_district\.csv\.gz$",
    ]
    if any(re.match(pattern, name) for pattern in basic_district_patterns):
        return "daily_od_district"

    if (
        re.search(r"^20\d{6}_", name)
        and ("viajes" in name or "trips" in name)
        and "complet" in name
        and name.endswith("csv.gz")
    ):
        return "complete_trip"

    is_district = "distrit" in name or "district" in name
    if (
        (
            "movilidad_obligada" in name
            or "obligated_mobility" in name
            or "mandatory_mobility" in name
            or "mobligated_mobility" in name
            or ("recurr" in name and "mobil" in name)
        )
        and is_district
        and name.endswith("csv.gz")
    ):
        return "recurrent_district"

    return "other"


catalog, catalog_source = load_official_catalog()
catalog["kind"] = catalog["filename"].map(classify_catalog_file)
catalog_snapshot = CATALOG_DIR / f"mitma_catalog_snapshot_{RUN_ID}.csv"
atomic_write_csv(catalog, catalog_snapshot)

print("Catalogue source:", catalog_source)
print(catalog["kind"].value_counts())
print("Snapshot:", catalog_snapshot)

## 5. Build the monthly typical-workweek schedule

In [ ]:
def get_national_holidays(years):
    if not EXCLUDE_NATIONAL_HOLIDAYS:
        return set()
    es_holidays = holidays.country_holidays("ES", years=years)
    return {pd.Timestamp(day).normalize() for day in es_holidays}


def select_typical_week(month, month_daily, national_holidays):
    period = pd.Period(month, freq="M")
    month_start = period.start_time.normalize()
    month_end = period.end_time.normalize()
    available = {
        pd.Timestamp(day).normalize()
        for day in pd.to_datetime(month_daily["file_date"].dropna())
    }

    candidate_mondays = pd.date_range(month_start, month_end, freq="W-MON")
    candidates = []

    for monday in candidate_mondays:
        dates = pd.date_range(monday, periods=5, freq="D")
        if dates[-1] > month_end:
            continue
        missing = [day for day in dates if day.normalize() not in available]
        holiday_dates = [day for day in dates if day.normalize() in national_holidays]
        midpoint = dates[2]
        candidates.append({
            "monday": monday,
            "friday": dates[-1],
            "dates": dates,
            "missing_dates": missing,
            "holiday_dates": holiday_dates,
            "distance_to_target": abs(midpoint.day - TARGET_DAY_OF_MONTH),
        })

    clean = [
        c for c in candidates
        if not c["missing_dates"] and not c["holiday_dates"]
    ]
    if clean:
        chosen = sorted(clean, key=lambda c: (c["distance_to_target"], c["monday"]))[0]
        reason = "closest_complete_holiday_free_week"
    else:
        complete = [c for c in candidates if not c["missing_dates"]]
        if complete and ALLOW_HOLIDAY_WEEK_FALLBACK:
            chosen = sorted(
                complete,
                key=lambda c: (len(c["holiday_dates"]), c["distance_to_target"], c["monday"]),
            )[0]
            reason = "holiday_week_fallback"
        else:
            return {
                "month": month,
                "status": "unavailable",
                "selection_reason": "no_complete_holiday_free_week",
                "week_start": None,
                "week_end": None,
                "selected_dates": None,
                "holiday_dates": None,
                "missing_dates": None,
            }

    return {
        "month": month,
        "status": "ready",
        "selection_reason": reason,
        "week_start": chosen["monday"].strftime("%Y-%m-%d"),
        "week_end": chosen["friday"].strftime("%Y-%m-%d"),
        "selected_dates": ";".join(day.strftime("%Y-%m-%d") for day in chosen["dates"]),
        "holiday_dates": ";".join(day.strftime("%Y-%m-%d") for day in chosen["holiday_dates"]),
        "missing_dates": ";".join(day.strftime("%Y-%m-%d") for day in chosen["missing_dates"]),
    }


all_months = month_range(START_MONTH, END_MONTH)
if MONTHS_TO_PROCESS is not None:
    requested = set(MONTHS_TO_PROCESS)
    all_months = [month for month in all_months if month in requested]

holiday_years = sorted({int(month[:4]) for month in all_months})
national_holidays = get_national_holidays(holiday_years)

plan_records = []
for month in all_months:
    month_daily = catalog.loc[
        (catalog["kind"] == "daily_od_district")
        & (catalog["file_month"] == month)
    ].copy()
    month_recurrent = catalog.loc[
        (catalog["kind"] == "recurrent_district")
        & (catalog["file_month"] == month)
    ].copy()

    record = select_typical_week(month, month_daily, national_holidays)
    record["daily_files_in_month"] = len(month_daily)
    record["recurrent_files"] = len(month_recurrent)
    if record["status"] == "ready" and month_recurrent.empty:
        record["status"] = "unavailable"
        record["selection_reason"] = "recurrent_file_missing"
    plan_records.append(record)

processing_plan = pd.DataFrame(plan_records)
plan_path = QC_DIR / "monthly_workweek_plan.csv"
atomic_write_csv(processing_plan, plan_path)

print("Months requested:", len(all_months))
print(processing_plan["status"].value_counts(dropna=False))
print("Plan saved:", plan_path)
display(processing_plan)

## 6. Download, field mapping, and daily/recurrent aggregation functions

In [ ]:
def download_file(url, destination, overwrite=False, chunk_size=1024 * 1024):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size > 0 and not overwrite:
        return destination, False

    partial_path = destination.with_suffix(destination.suffix + ".part")
    if partial_path.exists():
        partial_path.unlink()

    with SESSION.get(url, stream=True, timeout=300) as response:
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))
        with open(partial_path, "wb") as output, tqdm(
            total=total_size,
            unit="B",
            unit_scale=True,
            desc=destination.name,
        ) as progress:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    output.write(chunk)
                    progress.update(len(chunk))

    partial_path.replace(destination)
    return destination, True


def detect_delimiter(path):
    candidates = ["|", ";", ",", "\t"]
    with gzip.open(path, "rt", encoding="utf-8-sig", errors="replace") as file:
        header = file.readline()
    scores = {delimiter: header.count(delimiter) for delimiter in candidates}
    delimiter = max(scores, key=scores.get)
    if scores[delimiter] == 0:
        raise ValueError(f"Could not detect delimiter: {path}")
    return delimiter


def read_gzip_sample(path, nrows=20):
    delimiter = detect_delimiter(path)
    sample = pd.read_csv(
        path,
        sep=delimiter,
        compression="gzip",
        dtype=str,
        nrows=nrows,
        encoding="utf-8-sig",
        low_memory=False,
    )
    return sample, delimiter


COLUMN_ALIASES = {
    "date": ["date", "fecha"],
    "period": ["period", "periodo", "hour", "hora"],
    "origin": ["origin", "origen"],
    "destination": ["destination", "destino"],
    "activity_origin": ["activity_origin", "actividad_origen"],
    "activity_destination": ["activity_destination", "actividad_destino"],
    "study_possible_destination": [
        "study_possible_destination", "possible_study_destination",
        "estudio_destino_posible", "estudio_posible_destino",
    ],
    "age": ["age", "edad"],
    "sex": ["sex", "sexo", "gender"],
    "trips": ["trips", "trip", "travel", "viajes", "n_trips"],
    "month": ["month", "mes"],
    "recurrence": ["recurrence", "recurrencia"],
    "people": ["people", "persons", "personas"],
}


def map_columns(columns):
    actual = {normalise_token(column): column for column in columns}
    mapping = {}
    for canonical, aliases in COLUMN_ALIASES.items():
        mapping[canonical] = next(
            (
                actual[normalise_token(alias)]
                for alias in aliases
                if normalise_token(alias) in actual
            ),
            None,
        )
    return mapping


def qi(name):
    return '"' + str(name).replace('"', '""') + '"'


def qs(value):
    return "'" + str(value).replace("'", "''") + "'"


def sql_norm(column):
    return (
        "LOWER(REGEXP_REPLACE("
        "TRANSLATE("
        f"TRIM(CAST({qi(column)} AS VARCHAR)), "
        "'ÁÉÍÓÚÜÑáéíóúüñ', "
        "'AEIOUUNaeiouun'"
        "), "
        "'[^A-Za-z0-9<>+=./-]+', '_', 'g'))"
    )


def sql_in(expression, values):
    values_sql = ", ".join(qs(value) for value in sorted(values))
    return f"{expression} IN ({values_sql})"


HOME_VALUES = {"home", "casa", "household", "hogar"}
WORK_STUDY_VALUES = {
    "work_study", "workstudy", "work_or_study",
    "trabajo_estudio", "trabajo_o_estudio",
}
NON_STUDY_VALUES = {"no", "n", "false", "0"}
STUDY_VALUES = {"yes", "si", "s", "true", "1"}


def aggregate_basic_daily_district(file_path, columns):
    delimiter = detect_delimiter(file_path)
    required = ["origin", "destination", "activity_origin", "activity_destination", "trips"]
    missing = [name for name in required if columns.get(name) is None]
    if missing:
        raise KeyError(f"Missing daily columns: {missing}")

    study_column = columns.get("study_possible_destination")
    if REQUIRE_STUDY_FIELD and study_column is None:
        raise KeyError("The selected daily file has no study_possible_destination field.")

    activity_origin = sql_norm(columns["activity_origin"])
    activity_destination = sql_norm(columns["activity_destination"])
    home_origin = sql_in(activity_origin, HOME_VALUES)
    work_destination = sql_in(activity_destination, WORK_STUDY_VALUES)

    if study_column:
        study_expression = sql_norm(study_column)
        explicit_non_study = sql_in(study_expression, NON_STUDY_VALUES)
        explicit_study = sql_in(study_expression, STUDY_VALUES)
        non_study_condition = (
            explicit_non_study if REQUIRE_EXPLICIT_NON_STUDY else f"NOT ({explicit_study})"
        )
    else:
        non_study_condition = "TRUE"
        explicit_study = "FALSE"

    trips = (
        "TRY_CAST(REPLACE("
        f"CAST({qi(columns['trips'])} AS VARCHAR), ',', '.') AS DOUBLE)"
    )

    if USE_WORKING_AGE_FILTER:
        if columns.get("age") is None:
            raise KeyError("Age filter requested but age field is missing.")
        age_expression = sql_norm(columns["age"])
        age_values_sql = ", ".join(qs(normalise_token(v)) for v in sorted(WORKING_AGE_VALUES))
        age_where = f"{age_expression} IN ({age_values_sql})"
    else:
        age_where = "TRUE"

    file_date, _ = extract_file_temporal_metadata(file_path.name)
    date_text = file_date.strftime("%Y-%m-%d") if not pd.isna(file_date) else file_path.name[:8]

    query = f"""
    SELECT
        {qs(date_text)} AS date,
        CAST({qi(columns['origin'])} AS VARCHAR) AS origin,
        CAST({qi(columns['destination'])} AS VARCHAR) AS destination,
        SUM(COALESCE({trips}, 0)) AS total_trips,
        SUM(CASE WHEN {home_origin} AND {work_destination}
            THEN COALESCE({trips}, 0) ELSE 0 END) AS home_to_work_all,
        SUM(CASE WHEN {home_origin} AND {work_destination} AND {non_study_condition}
            THEN COALESCE({trips}, 0) ELSE 0 END) AS home_to_work_nonstudy,
        SUM(CASE WHEN {home_origin} AND {work_destination} AND {explicit_study}
            THEN COALESCE({trips}, 0) ELSE 0 END) AS possible_education_trips
    FROM read_csv_auto(
        {qs(file_path.as_posix())},
        delim={qs(delimiter)},
        header=TRUE,
        all_varchar=TRUE,
        ignore_errors=TRUE
    )
    WHERE {age_where}
    GROUP BY origin, destination
    """

    connection = duckdb.connect()
    try:
        result = connection.execute(query).df()
    finally:
        connection.close()

    result["origin"] = result["origin"].map(canonical_district_id)
    result["destination"] = result["destination"].map(canonical_district_id)
    result = result.dropna(subset=["origin", "destination"])
    return result


def aggregate_recurrent_district(file_path, columns, target_month):
    delimiter = detect_delimiter(file_path)
    recurrence_band = f"TRIM(CAST({qi(columns['recurrence'])} AS VARCHAR))"
    people = (
        "TRY_CAST(REPLACE("
        f"CAST({qi(columns['people'])} AS VARCHAR), ',', '.') AS DOUBLE)"
    )

    if USE_WORKING_AGE_FILTER:
        if columns.get("age") is None:
            raise KeyError("Age filter requested but recurrent age is missing.")
        age_expression = sql_norm(columns["age"])
        age_values_sql = ", ".join(qs(normalise_token(v)) for v in sorted(WORKING_AGE_VALUES))
        age_where = f"{age_expression} IN ({age_values_sql})"
    else:
        age_where = "TRUE"

    allowed_bands_sql = ", ".join(qs(band) for band in RECURRENCE_BAND_ORDER)

    query = f"""
    SELECT
        {qs(target_month)} AS month,
        CAST({qi(columns['origin'])} AS VARCHAR) AS origin,
        CAST({qi(columns['destination'])} AS VARCHAR) AS destination,
        {recurrence_band} AS recurrence_band,
        SUM(COALESCE({people}, 0)) AS people
    FROM read_csv_auto(
        {qs(file_path.as_posix())},
        delim={qs(delimiter)},
        header=TRUE,
        all_varchar=TRUE,
        ignore_errors=TRUE
    )
    WHERE {age_where} AND {recurrence_band} IN ({allowed_bands_sql})
    GROUP BY origin, destination, recurrence_band
    """

    connection = duckdb.connect()
    try:
        result = connection.execute(query).df()
    finally:
        connection.close()

    result["origin"] = result["origin"].map(canonical_district_id)
    result["destination"] = result["destination"].map(canonical_district_id)
    result = result.dropna(subset=["origin", "destination"])
    return result

## 7. Optional district-centroid matching

In [ ]:
def modal_numeric_length(values):
    lengths = [len(str(value)) for value in values if value is not None and str(value).isdigit()]
    if not lengths:
        return None
    return int(pd.Series(lengths).mode().iloc[0])


def candidate_id_variants(series, target_length=None):
    canonical = series.map(canonical_district_id)
    digits = canonical.str.replace(r"\D", "", regex=True)
    variants = {"canonical": canonical, "digits_only": digits}
    if target_length:
        variants["canonical_zfill"] = canonical.map(
            lambda value: value.zfill(target_length)
            if value is not None and str(value).isdigit() else value
        )
        variants["digits_zfill"] = digits.map(
            lambda value: value.zfill(target_length)
            if value is not None and str(value).isdigit() else value
        )
    return variants


def score_id_fields(table, target_ids, geometry_column=None):
    target_ids = set(target_ids)
    target_length = modal_numeric_length(target_ids)
    records = []

    for column in table.columns:
        if geometry_column is not None and column == geometry_column:
            continue
        for variant_name, values in candidate_id_variants(
            table[column], target_length=target_length
        ).items():
            unique_values = set(values.dropna().unique())
            intersection = unique_values & target_ids
            records.append({
                "field": column,
                "variant": variant_name,
                "match_count": len(intersection),
                "target_match_share": len(intersection) / len(target_ids) if target_ids else np.nan,
                "field_unique_match_share": (
                    len(intersection) / len(unique_values) if unique_values else np.nan
                ),
                "field_unique_values": len(unique_values),
            })

    return (
        pd.DataFrame(records)
        .sort_values(
            ["target_match_share", "field_unique_match_share", "match_count"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )


CENTROIDS_RAW = None
CENTROID_LOOKUP = None
CENTROID_MATCH_INFO = None

if ADD_DISTANCE_IF_AVAILABLE and DISTRICT_CENTROID_SHP.exists():
    import geopandas as gpd
    CENTROIDS_RAW = gpd.read_file(DISTRICT_CENTROID_SHP)
    if CENTROIDS_RAW.crs is None:
        raise ValueError("District centroid CRS is missing.")
    print(f"Loaded {len(CENTROIDS_RAW):,} centroid features. ID matching will use the first monthly network.")
else:
    print("Centroid file unavailable; distance_km will not be added.")


def ensure_centroid_lookup(network_ids):
    global CENTROID_LOOKUP, CENTROID_MATCH_INFO

    if CENTROIDS_RAW is None:
        return None, "Centroid file unavailable."
    if CENTROID_LOOKUP is not None:
        return CENTROID_LOOKUP, CENTROID_MATCH_INFO

    target_ids = set(network_ids)
    geometry_column = CENTROIDS_RAW.geometry.name
    scores = score_id_fields(CENTROIDS_RAW, target_ids, geometry_column=geometry_column)
    if scores.empty or int(scores.iloc[0]["match_count"]) == 0:
        return None, "No centroid field matched the network district IDs."

    best = scores.iloc[0]
    target_length = modal_numeric_length(target_ids)
    variants = candidate_id_variants(
        CENTROIDS_RAW[best["field"]], target_length=target_length
    )

    centroids = CENTROIDS_RAW.copy()
    centroids["district_id"] = variants[best["variant"]]
    centroids = (
        centroids.dropna(subset=["district_id", geometry_column])
        .drop_duplicates("district_id")
        .to_crs(DISTANCE_CRS)
    )
    centroids["x"] = centroids.geometry.x
    centroids["y"] = centroids.geometry.y
    lookup = centroids.set_index("district_id")[["x", "y"]]

    match_share = float(best["target_match_share"])
    scores.to_csv(
        QC_DIR / "centroid_id_field_scores.csv", index=False, encoding="utf-8-sig"
    )
    match_info = {
        "field": str(best["field"]),
        "variant": str(best["variant"]),
        "matched_network_ids": int(best["match_count"]),
        "network_match_share": match_share,
    }
    atomic_write_json(match_info, QC_DIR / "centroid_id_match.json")

    if match_share < MIN_CENTROID_NODE_MATCH_SHARE:
        return None, (
            f"Best centroid match share was {match_share:.3f}, below the required "
            f"{MIN_CENTROID_NODE_MATCH_SHARE:.3f}; distance was not added."
        )

    CENTROID_LOOKUP = lookup
    CENTROID_MATCH_INFO = (
        f"Centroid IDs matched using {best['field']} / {best['variant']}; "
        f"network match share {match_share:.3f}."
    )
    return CENTROID_LOOKUP, CENTROID_MATCH_INFO

## 8. Monthly paths, completion checks, and network allocation

In [ ]:
FINAL_EDGE_REQUIRED_COLUMNS = {
    "month",
    "week_start",
    "week_end",
    "origin",
    "destination",
    "self_loop",
    "mean_weekday_work_trips",
    *LAYER_WEIGHT_COLUMNS.values(),
    "allocation_total",
    "allocation_error",
}

FINAL_LONG_REQUIRED_COLUMNS = {
    "month",
    "week_start",
    "week_end",
    "origin",
    "destination",
    "self_loop",
    "recurrence_band",
    "recurrence_midpoint",
    "recurrence_share_14d",
    "hybrid_intensity_score",
    "hybrid_rank_score",
    "layer_weight",
}

FINAL_INTENSITY_REQUIRED_COLUMNS = {
    "month",
    "district_id",
    "role",
    "total_layer_flow",
    "hybrid_intensity",
    "mean_recurrence_midpoint",
}


def month_paths(month):
    safe = month.replace("-", "_")

    return {
        "temp": TEMP_DIR / month,
        "daily_dir": DAILY_PARTIAL_DIR / month,
        "workweek_od": WORKWEEK_OD_DIR / f"workweek_od_{safe}.parquet",
        "recurrent": RECURRENT_DIR / f"recurrent_{safe}.parquet",
        "edge": LAYER_NETWORK_DIR / f"network_layer_edges_{safe}.parquet",
        "long": LAYER_LONG_DIR / f"network_layer_edges_long_{safe}.parquet",
        "node_intensity": (
            NODE_INTENSITY_DIR
            / f"district_hybrid_intensity_{safe}.parquet"
        ),
        "qc": QC_DIR / "monthly" / f"monthly_qc_{safe}.csv",
        "success": MONTH_STATUS_DIR / f"{month}_SUCCESS.json",
        "failure": MONTH_STATUS_DIR / f"{month}_FAILED.json",
        "manifest": MONTH_STATUS_DIR / f"{month}_manifest.json",
    }


def remove_month_outputs(month, include_intermediate=True):
    paths = month_paths(month)

    for key in [
        "edge",
        "long",
        "node_intensity",
        "qc",
        "success",
        "failure",
        "manifest",
    ]:
        path = paths[key]

        if path.exists():
            path.unlink()

    if include_intermediate:
        for key in ["workweek_od", "recurrent"]:
            path = paths[key]

            if path.exists():
                path.unlink()

        for key in ["daily_dir", "temp"]:
            path = paths[key]

            if path.exists():
                shutil.rmtree(path)


def month_is_complete(month):
    paths = month_paths(month)
    marker = safe_read_json(paths["success"])

    if marker is None:
        return False, "success_marker_missing"

    if (
        REPROCESS_IF_CONFIG_CHANGED
        and marker.get("config_signature") != CONFIG_SIGNATURE
    ):
        return False, "configuration_changed"

    if not validate_parquet(
        paths["edge"],
        FINAL_EDGE_REQUIRED_COLUMNS,
        min_rows=1,
    ):
        return False, "wide_edge_table_invalid"

    if not validate_parquet(
        paths["long"],
        FINAL_LONG_REQUIRED_COLUMNS,
        min_rows=1,
    ):
        return False, "long_edge_table_invalid"

    if not validate_parquet(
        paths["node_intensity"],
        FINAL_INTENSITY_REQUIRED_COLUMNS,
        min_rows=1,
    ):
        return False, "node_intensity_table_invalid"

    if not paths["qc"].exists():
        return False, "monthly_qc_missing"

    return True, "complete"


def add_distance(edges):
    network_ids = (
        set(edges["origin"].dropna())
        | set(edges["destination"].dropna())
    )

    lookup, message = ensure_centroid_lookup(network_ids)

    if lookup is None:
        print(message)

        result = edges.copy()
        result["distance_km"] = np.nan

        return result, False, np.nan

    result = (
        edges.merge(
            lookup.rename(
                columns={
                    "x": "origin_x",
                    "y": "origin_y",
                }
            ),
            left_on="origin",
            right_index=True,
            how="left",
        )
        .merge(
            lookup.rename(
                columns={
                    "x": "destination_x",
                    "y": "destination_y",
                }
            ),
            left_on="destination",
            right_index=True,
            how="left",
        )
    )

    matched = (
        result[
            [
                "origin_x",
                "origin_y",
                "destination_x",
                "destination_y",
            ]
        ]
        .notna()
        .all(axis=1)
    )

    result["distance_km"] = np.nan

    result.loc[matched, "distance_km"] = (
        np.sqrt(
            (
                result.loc[matched, "destination_x"]
                - result.loc[matched, "origin_x"]
            ) ** 2
            + (
                result.loc[matched, "destination_y"]
                - result.loc[matched, "origin_y"]
            ) ** 2
        )
        / 1000.0
    )

    result.loc[
        result["self_loop"],
        "distance_km",
    ] = 0.0

    match_share = (
        matched.mean()
        if len(result)
        else np.nan
    )

    result = result.drop(
        columns=[
            "origin_x",
            "origin_y",
            "destination_x",
            "destination_y",
        ]
    )

    return result, True, match_share


def build_network_edges(
    month,
    week_start,
    week_end,
    workweek_od,
    recurrent_district,
):
    recurrent = recurrent_district.copy()

    recurrent["recurrence_midpoint"] = (
        recurrent["recurrence_band"]
        .map(RECURRENCE_MIDPOINT)
    )

    recurrent["tripday_weight"] = (
        recurrent["people"]
        * recurrent["recurrence_midpoint"]
    )

    recurrent_totals = (
        recurrent.groupby(
            ["origin", "destination"],
            as_index=False,
        )
        .agg(
            recurrent_people_total=(
                "people",
                "sum",
            ),
            tripday_weight_total=(
                "tripday_weight",
                "sum",
            ),
            recurrence_bands_available=(
                "recurrence_band",
                "nunique",
            ),
        )
    )

    recurrent = recurrent.merge(
        recurrent_totals,
        on=["origin", "destination"],
        how="left",
        validate="many_to_one",
    )

    recurrent["tripday_share"] = np.where(
        recurrent["tripday_weight_total"] > 0,
        (
            recurrent["tripday_weight"]
            / recurrent["tripday_weight_total"]
        ),
        np.nan,
    )

    work_columns = [
        "origin",
        "destination",
        "observed_days",
        "active_work_days",
        "mean_weekday_work_trips",
    ]

    work = (
        workweek_od[work_columns]
        .drop_duplicates(
            ["origin", "destination"]
        )
    )

    fused = recurrent.merge(
        work,
        on=["origin", "destination"],
        how="inner",
        validate="many_to_one",
    )

    fused["allocated_weight"] = (
        fused["mean_weekday_work_trips"]
        * fused["tripday_share"]
    )

    edge_base = work.merge(
        recurrent_totals,
        on=["origin", "destination"],
        how="inner",
        validate="one_to_one",
    )

    for band in RECURRENCE_BAND_ORDER:
        output_column = LAYER_WEIGHT_COLUMNS[band]

        values = (
            fused.loc[
                fused["recurrence_band"] == band
            ]
            .groupby(
                ["origin", "destination"],
                as_index=False,
            )
            .agg(
                **{
                    output_column: (
                        "allocated_weight",
                        "sum",
                    )
                }
            )
        )

        edge_base = edge_base.merge(
            values,
            on=["origin", "destination"],
            how="left",
            validate="one_to_one",
        )

    layer_columns = list(
        LAYER_WEIGHT_COLUMNS.values()
    )

    edge_base[layer_columns] = (
        edge_base[layer_columns]
        .fillna(0.0)
    )

    edge_base["allocation_total"] = (
        edge_base[layer_columns]
        .sum(axis=1)
    )

    edge_base["allocation_error"] = (
        edge_base["allocation_total"]
        - edge_base["mean_weekday_work_trips"]
    )

    # Legacy aggregate columns. Subsequent notebooks do not use them.
    edge_base["low_frequency_weight"] = (
        edge_base[
            LAYER_WEIGHT_COLUMNS["1-2"]
        ]
    )

    edge_base["hybrid_weight"] = (
        edge_base[
            [
                LAYER_WEIGHT_COLUMNS["3-4"],
                LAYER_WEIGHT_COLUMNS["5-7"],
            ]
        ]
        .sum(axis=1)
    )

    edge_base["onsite_standard_weight"] = (
        edge_base[
            LAYER_WEIGHT_COLUMNS["8-10"]
        ]
    )

    edge_base["onsite_very_high_weight"] = (
        edge_base[
            [
                LAYER_WEIGHT_COLUMNS["11-13"],
                LAYER_WEIGHT_COLUMNS["14"],
            ]
        ]
        .sum(axis=1)
    )

    edge_base["onsite_broad_weight"] = (
        edge_base[
            [
                LAYER_WEIGHT_COLUMNS["8-10"],
                LAYER_WEIGHT_COLUMNS["11-13"],
                LAYER_WEIGHT_COLUMNS["14"],
            ]
        ]
        .sum(axis=1)
    )

    edge_base["month"] = month
    edge_base["week_start"] = week_start
    edge_base["week_end"] = week_end
    edge_base["self_loop"] = (
        edge_base["origin"]
        == edge_base["destination"]
    )

    final_columns = [
        "month",
        "week_start",
        "week_end",
        "origin",
        "destination",
        "self_loop",
        "observed_days",
        "active_work_days",
        "mean_weekday_work_trips",
        "recurrent_people_total",
        "recurrence_bands_available",
        *layer_columns,
        "allocation_total",
        "allocation_error",
        "low_frequency_weight",
        "hybrid_weight",
        "onsite_standard_weight",
        "onsite_very_high_weight",
        "onsite_broad_weight",
    ]

    return (
        edge_base[final_columns].copy(),
        recurrent_totals,
    )


def build_long_layer_edges(wide_edges):
    identifier_columns = [
        "month",
        "week_start",
        "week_end",
        "origin",
        "destination",
        "self_loop",
        "observed_days",
        "active_work_days",
        "mean_weekday_work_trips",
        "distance_km",
    ]

    identifier_columns = [
        column
        for column in identifier_columns
        if column in wide_edges.columns
    ]

    reverse_weight_columns = {
        column: band
        for band, column in (
            LAYER_WEIGHT_COLUMNS.items()
        )
    }

    long_edges = wide_edges.melt(
        id_vars=identifier_columns,
        value_vars=list(
            reverse_weight_columns
        ),
        var_name="layer_weight_column",
        value_name="layer_weight",
    )

    long_edges["recurrence_band"] = (
        long_edges["layer_weight_column"]
        .map(reverse_weight_columns)
    )

    long_edges["recurrence_midpoint"] = (
        long_edges["recurrence_band"]
        .map(RECURRENCE_MIDPOINT)
    )

    long_edges["recurrence_share_14d"] = (
        long_edges["recurrence_midpoint"]
        / 14.0
    )

    long_edges["hybrid_intensity_score"] = (
        long_edges["recurrence_band"]
        .map(HYBRID_INTENSITY_SCORE)
    )

    long_edges["hybrid_rank_score"] = (
        long_edges["recurrence_band"]
        .map(HYBRID_RANK_SCORE)
    )

    long_edges["layer_order_recurrence"] = (
        long_edges["recurrence_band"]
        .map({
            band: index
            for index, band in enumerate(
                RECURRENCE_BAND_ORDER
            )
        })
    )

    long_edges["layer_order_hybrid"] = (
        long_edges["recurrence_band"]
        .map({
            band: index
            for index, band in enumerate(
                reversed(
                    RECURRENCE_BAND_ORDER
                )
            )
        })
    )

    long_edges["layer_label"] = (
        long_edges["recurrence_band"]
        .map(RECURRENCE_LAYER_LABEL)
    )

    long_edges = long_edges.loc[
        long_edges["layer_weight"] > 0
    ].copy()

    return long_edges.drop(
        columns="layer_weight_column"
    )


def calculate_node_hybrid_intensity(
    long_edges,
    role,
):
    if role == "residential":
        node_column = "origin"
    elif role == "employment":
        node_column = "destination"
    else:
        raise ValueError(
            "role must be residential "
            "or employment"
        )

    working = long_edges[
        [
            "month",
            node_column,
            "recurrence_band",
            "recurrence_midpoint",
            "hybrid_intensity_score",
            "layer_weight",
        ]
    ].copy()

    working["weighted_hybrid_score"] = (
        working["layer_weight"]
        * working["hybrid_intensity_score"]
    )

    working["weighted_recurrence_midpoint"] = (
        working["layer_weight"]
        * working["recurrence_midpoint"]
    )

    summary = (
        working.groupby(
            ["month", node_column],
            as_index=False,
        )
        .agg(
            total_layer_flow=(
                "layer_weight",
                "sum",
            ),
            weighted_hybrid_score=(
                "weighted_hybrid_score",
                "sum",
            ),
            weighted_recurrence_midpoint=(
                "weighted_recurrence_midpoint",
                "sum",
            ),
            observed_recurrence_layers=(
                "recurrence_band",
                "nunique",
            ),
        )
        .rename(
            columns={
                node_column: "district_id",
            }
        )
    )

    summary["hybrid_intensity"] = (
        summary["weighted_hybrid_score"]
        / summary["total_layer_flow"]
    )

    summary["mean_recurrence_midpoint"] = (
        summary["weighted_recurrence_midpoint"]
        / summary["total_layer_flow"]
    )

    band_flow = (
        working.groupby(
            [
                "month",
                node_column,
                "recurrence_band",
            ],
            as_index=False,
        )["layer_weight"]
        .sum()
    )

    band_flow["band_share"] = (
        band_flow["layer_weight"]
        / band_flow.groupby(
            ["month", node_column]
        )["layer_weight"]
        .transform("sum")
    )

    band_share_wide = (
        band_flow.pivot(
            index=["month", node_column],
            columns="recurrence_band",
            values="band_share",
        )
        .reset_index()
    )

    band_share_wide = band_share_wide.rename(
        columns={
            node_column: "district_id",
            **{
                band: (
                    "share_recurrence_"
                    + band.replace("-", "_")
                )
                for band in RECURRENCE_BAND_ORDER
            },
        }
    )

    summary = summary.merge(
        band_share_wide,
        on=["month", "district_id"],
        how="left",
        validate="one_to_one",
    )

    summary["role"] = role

    return summary.drop(
        columns=[
            "weighted_hybrid_score",
            "weighted_recurrence_midpoint",
        ]
    )


def build_node_intensity_table(long_edges):
    residential = (
        calculate_node_hybrid_intensity(
            long_edges,
            role="residential",
        )
    )

    employment = (
        calculate_node_hybrid_intensity(
            long_edges,
            role="employment",
        )
    )

    return pd.concat(
        [residential, employment],
        ignore_index=True,
    )

## 9. Monthly processing with daily and stage-level checkpoints

In [ ]:
def process_month(month, plan_row):
    month_start_clock = time.perf_counter()
    paths = month_paths(month)
    force = month in FORCE_REPROCESS_MONTHS

    if force:
        print(f"[{month}] Forced reprocessing requested.")
        remove_month_outputs(month, include_intermediate=True)

    complete, complete_reason = month_is_complete(month)
    if complete_reason == "configuration_changed" and REPROCESS_IF_CONFIG_CHANGED and not force:
        print(f"[{month}] Configuration changed; clearing old monthly outputs.")
        remove_month_outputs(month, include_intermediate=True)
        complete = False
        complete_reason = "cleared_after_configuration_change"

    if SKIP_COMPLETED_MONTHS and complete and not force:
        print(f"[{month}] SKIP — already complete.")
        return {
            "month": month,
            "run_id": RUN_ID,
            "status": "skipped_complete",
            "message": complete_reason,
            "elapsed_minutes": 0.0,
            "config_signature": CONFIG_SIGNATURE,
            "updated_at": datetime.now().isoformat(timespec="seconds"),
        }

    if plan_row["status"] != "ready":
        raise RuntimeError(f"Month unavailable in plan: {plan_row['selection_reason']}")

    for key in ["temp", "daily_dir"]:
        paths[key].mkdir(parents=True, exist_ok=True)
    paths["qc"].parent.mkdir(parents=True, exist_ok=True)

    selected_dates = [pd.Timestamp(value) for value in str(plan_row["selected_dates"]).split(";")]
    week_start = str(plan_row["week_start"])
    week_end = str(plan_row["week_end"])

    previous_manifest = safe_read_json(paths["manifest"])
    new_selected_dates = [day.strftime("%Y-%m-%d") for day in selected_dates]
    if previous_manifest is not None:
        manifest_changed = (
            previous_manifest.get("selected_dates") != new_selected_dates
            or previous_manifest.get("config_signature") != CONFIG_SIGNATURE
        )
        if manifest_changed:
            print(f"[{month}] Workweek/config manifest changed; clearing partial daily and workweek outputs.")
            if paths["daily_dir"].exists():
                shutil.rmtree(paths["daily_dir"])
            if paths["workweek_od"].exists():
                paths["workweek_od"].unlink()
            paths["daily_dir"].mkdir(parents=True, exist_ok=True)

    manifest = {
        "month": month,
        "week_start": week_start,
        "week_end": week_end,
        "selected_dates": new_selected_dates,
        "selection_reason": plan_row["selection_reason"],
        "config_signature": CONFIG_SIGNATURE,
        "run_id_last_touched": RUN_ID,
    }
    atomic_write_json(manifest, paths["manifest"])

    month_daily = (
        catalog.loc[
            (catalog["kind"] == "daily_od_district")
            & (catalog["file_month"] == month)
        ]
        .sort_values("file_date")
        .drop_duplicates("file_date")
        .copy()
    )
    month_daily["date"] = pd.to_datetime(month_daily["file_date"]).dt.normalize()
    selected_daily = month_daily.loc[month_daily["date"].isin(selected_dates)].copy()

    if len(selected_daily) != EXPECTED_WEEKDAYS:
        raise RuntimeError(
            f"Expected {EXPECTED_WEEKDAYS} daily files, found {len(selected_daily)} for {month}."
        )

    month_recurrent = (
        catalog.loc[
            (catalog["kind"] == "recurrent_district")
            & (catalog["file_month"] == month)
        ]
        .sort_values("filename")
        .copy()
    )
    if month_recurrent.empty:
        raise FileNotFoundError(f"No recurrent district file for {month}.")

    # --------------------------------------------------------
    # Stage A: daily files. Each daily summary is a checkpoint.
    # --------------------------------------------------------
    daily_paths = []
    for row in selected_daily.itertuples(index=False):
        day_text = row.date.strftime("%Y%m%d")
        summary_path = paths["daily_dir"] / f"{day_text}_daily_work_od.parquet"
        raw_path = paths["temp"] / row.filename

        daily_required = {
            "date", "origin", "destination", "home_to_work_nonstudy"
        }
        if (
            REUSE_PARTIAL_OUTPUTS
            and not force
            and validate_parquet(summary_path, daily_required, min_rows=1)
        ):
            print(f"[{month}] Reuse daily summary {day_text}.")
            daily_paths.append(summary_path)
            continue

        started = time.perf_counter()
        raw_path, downloaded_now = download_file(row.link, raw_path, overwrite=force)
        download_seconds = time.perf_counter() - started
        raw_size = file_size_mb(raw_path)

        if "complet" in raw_path.name.lower():
            raise RuntimeError(f"Wrong daily product selected: {raw_path.name}")

        sample, _ = read_gzip_sample(raw_path)
        columns = map_columns(sample.columns)
        processing_started = time.perf_counter()
        summary = aggregate_basic_daily_district(raw_path, columns)
        temp_summary = summary_path.with_suffix(".parquet.tmp")
        summary.to_parquet(temp_summary, index=False, compression="zstd")
        temp_summary.replace(summary_path)
        processing_seconds = time.perf_counter() - processing_started

        if not KEEP_RAW_DAILY and raw_path.exists():
            raw_path.unlink()

        upsert_csv({
            "month": month,
            "date": row.date.strftime("%Y-%m-%d"),
            "run_id": RUN_ID,
            "downloaded_now": downloaded_now,
            "download_seconds": download_seconds,
            "processing_seconds": processing_seconds,
            "raw_size_mb": raw_size,
            "summary_size_mb": file_size_mb(summary_path),
            "summary_rows": len(summary),
            "updated_at": datetime.now().isoformat(timespec="seconds"),
        }, QC_DIR / "daily_file_timings.csv", keys=["month", "date"])

        print(
            f"[{month}] {row.date:%Y-%m-%d}: "
            f"download {download_seconds/60:.2f} min; "
            f"process {processing_seconds/60:.2f} min; {len(summary):,} rows."
        )
        daily_paths.append(summary_path)

    # --------------------------------------------------------
    # Stage B: workweek OD. Reused if valid.
    # --------------------------------------------------------
    workweek_required = {
        "origin", "destination", "observed_days",
        "active_work_days", "mean_weekday_work_trips",
    }
    if (
        REUSE_PARTIAL_OUTPUTS
        and not force
        and validate_parquet(paths["workweek_od"], workweek_required, min_rows=1)
    ):
        workweek_od = pd.read_parquet(paths["workweek_od"])
        print(f"[{month}] Reuse workweek OD.")
    else:
        daily = pd.concat([pd.read_parquet(path) for path in daily_paths], ignore_index=True)
        daily["date"] = pd.to_datetime(daily["date"])
        daily["active_work_day"] = daily["home_to_work_nonstudy"] > ACTIVE_DAY_WORK_TRIP_THRESHOLD

        workweek_od = (
            daily.groupby(["origin", "destination"], as_index=False)
            .agg(
                work_trips=("home_to_work_nonstudy", "sum"),
                observed_days=("date", "nunique"),
                active_work_days=("active_work_day", "sum"),
                home_to_work_all=("home_to_work_all", "sum"),
                possible_education_trips=("possible_education_trips", "sum"),
            )
        )
        workweek_od["mean_weekday_work_trips"] = workweek_od["work_trips"] / EXPECTED_WEEKDAYS
        workweek_od["possible_education_share"] = np.where(
            workweek_od["home_to_work_all"] > 0,
            workweek_od["possible_education_trips"] / workweek_od["home_to_work_all"],
            0.0,
        )
        workweek_od = workweek_od.loc[
            workweek_od["mean_weekday_work_trips"] > MIN_MEAN_WEEKDAY_WORK_TRIPS
        ].copy()
        temp_path = paths["workweek_od"].with_suffix(".parquet.tmp")
        workweek_od.to_parquet(temp_path, index=False, compression="zstd")
        temp_path.replace(paths["workweek_od"])

    # --------------------------------------------------------
    # Stage C: recurrent matrix. Reused if valid.
    # --------------------------------------------------------
    recurrent_required = {"month", "origin", "destination", "recurrence_band", "people"}
    if (
        REUSE_PARTIAL_OUTPUTS
        and not force
        and validate_parquet(paths["recurrent"], recurrent_required, min_rows=1)
    ):
        recurrent = pd.read_parquet(paths["recurrent"])
        print(f"[{month}] Reuse recurrent matrix.")
    else:
        recurrent_parts = []
        for row in month_recurrent.itertuples(index=False):
            raw_path = paths["temp"] / row.filename
            download_started = time.perf_counter()
            raw_path, downloaded_now = download_file(row.link, raw_path, overwrite=force)
            download_seconds = time.perf_counter() - download_started
            raw_size = file_size_mb(raw_path)

            sample, _ = read_gzip_sample(raw_path)
            columns = map_columns(sample.columns)
            required = ["origin", "destination", "recurrence", "people"]
            missing = [name for name in required if columns.get(name) is None]
            if missing:
                raise KeyError(f"Missing recurrent columns for {month}: {missing}")

            processing_started = time.perf_counter()
            part = aggregate_recurrent_district(raw_path, columns, month)
            processing_seconds = time.perf_counter() - processing_started
            recurrent_parts.append(part)

            upsert_csv({
                "month": month,
                "filename": row.filename,
                "run_id": RUN_ID,
                "downloaded_now": downloaded_now,
                "download_seconds": download_seconds,
                "processing_seconds": processing_seconds,
                "raw_size_mb": raw_size,
                "rows_after_aggregation": len(part),
                "updated_at": datetime.now().isoformat(timespec="seconds"),
            }, QC_DIR / "recurrent_file_timings.csv", keys=["month", "filename"])

            if not KEEP_RAW_RECURRENT and raw_path.exists():
                raw_path.unlink()

        recurrent = (
            pd.concat(recurrent_parts, ignore_index=True)
            .groupby(["month", "origin", "destination", "recurrence_band"], as_index=False)
            .agg(people=("people", "sum"))
        )
        temp_path = paths["recurrent"].with_suffix(".parquet.tmp")
        recurrent.to_parquet(temp_path, index=False, compression="zstd")
        temp_path.replace(paths["recurrent"])


    # --------------------------------------------------------
    # Stage D: six-layer network, long table, node intensity and QC.
    # --------------------------------------------------------
    edges, recurrent_totals = build_network_edges(
        month,
        week_start,
        week_end,
        workweek_od,
        recurrent,
    )

    edges, distance_added, distance_match_share = add_distance(edges)

    long_edges = build_long_layer_edges(edges)
    node_intensity = build_node_intensity_table(long_edges)

    temp_edge = paths["edge"].with_suffix(".parquet.tmp")
    edges.to_parquet(
        temp_edge,
        index=False,
        compression="zstd",
    )
    temp_edge.replace(paths["edge"])

    temp_long = paths["long"].with_suffix(".parquet.tmp")
    long_edges.to_parquet(
        temp_long,
        index=False,
        compression="zstd",
    )
    temp_long.replace(paths["long"])

    temp_intensity = paths["node_intensity"].with_suffix(
        ".parquet.tmp"
    )
    node_intensity.to_parquet(
        temp_intensity,
        index=False,
        compression="zstd",
    )
    temp_intensity.replace(paths["node_intensity"])

    if not validate_parquet(
        paths["edge"],
        FINAL_EDGE_REQUIRED_COLUMNS,
        min_rows=1,
    ):
        raise RuntimeError(
            f"Wide layer-edge validation failed for {month}."
        )

    if not validate_parquet(
        paths["long"],
        FINAL_LONG_REQUIRED_COLUMNS,
        min_rows=1,
    ):
        raise RuntimeError(
            f"Long layer-edge validation failed for {month}."
        )

    if not validate_parquet(
        paths["node_intensity"],
        FINAL_INTENSITY_REQUIRED_COLUMNS,
        min_rows=1,
    ):
        raise RuntimeError(
            f"Node-intensity validation failed for {month}."
        )

    recurrent_keys = (
        recurrent_totals[
            ["origin", "destination"]
        ]
        .copy()
    )
    recurrent_keys["matched"] = True

    match_check = workweek_od.merge(
        recurrent_keys,
        on=["origin", "destination"],
        how="left",
    )

    match_check["matched"] = (
        match_check["matched"]
        .fillna(False)
    )

    total_flow = (
        workweek_od[
            "mean_weekday_work_trips"
        ]
        .sum()
    )

    matched_flow = (
        match_check.loc[
            match_check["matched"],
            "mean_weekday_work_trips",
        ]
        .sum()
    )

    layer_totals = {
        (
            "total_weight_"
            + band.replace("-", "_")
        ): float(
            edges[
                LAYER_WEIGHT_COLUMNS[band]
            ].sum()
        )
        for band in RECURRENCE_BAND_ORDER
    }

    layer_self_loop_shares = {
        (
            "self_loop_share_"
            + band.replace("-", "_")
        ): (
            float(
                edges.loc[
                    edges["self_loop"],
                    LAYER_WEIGHT_COLUMNS[band],
                ].sum()
                / edges[
                    LAYER_WEIGHT_COLUMNS[band]
                ].sum()
            )
            if edges[
                LAYER_WEIGHT_COLUMNS[band]
            ].sum() > 0
            else np.nan
        )
        for band in RECURRENCE_BAND_ORDER
    }

    month_elapsed = (
        time.perf_counter()
        - month_start_clock
    )

    qc_payload = {
        "month": month,
        "week_start": week_start,
        "week_end": week_end,
        "selection_reason": plan_row["selection_reason"],
        "config_signature": CONFIG_SIGNATURE,
        "positive_work_od": len(workweek_od),
        "matched_work_od": int(
            match_check["matched"].sum()
        ),
        "od_match_share": float(
            match_check["matched"].mean()
        ),
        "weekday_flow_total": total_flow,
        "weekday_flow_matched": matched_flow,
        "weekday_flow_match_share": (
            matched_flow / total_flow
            if total_flow > 0
            else np.nan
        ),
        "wide_edge_rows": len(edges),
        "long_layer_edge_rows": len(long_edges),
        "node_intensity_rows": len(node_intensity),
        "max_abs_allocation_error": float(
            edges["allocation_error"]
            .abs()
            .max()
        ),
        "distance_added": distance_added,
        "distance_match_share": distance_match_share,
        "workweek_od_size_mb": file_size_mb(
            paths["workweek_od"]
        ),
        "recurrent_size_mb": file_size_mb(
            paths["recurrent"]
        ),
        "wide_edge_size_mb": file_size_mb(
            paths["edge"]
        ),
        "long_edge_size_mb": file_size_mb(
            paths["long"]
        ),
        "node_intensity_size_mb": file_size_mb(
            paths["node_intensity"]
        ),
        "elapsed_minutes": month_elapsed / 60.0,
        "completed_at": datetime.now().isoformat(
            timespec="seconds"
        ),
        **layer_totals,
        **layer_self_loop_shares,
    }

    qc = pd.DataFrame([qc_payload])
    atomic_write_csv(qc, paths["qc"])

    success_payload = {
        "month": month,
        "status": "success",
        "config_signature": CONFIG_SIGNATURE,
        "run_id": RUN_ID,
        "completed_at": datetime.now().isoformat(
            timespec="seconds"
        ),
        "wide_edge_file": str(paths["edge"]),
        "long_edge_file": str(paths["long"]),
        "node_intensity_file": str(
            paths["node_intensity"]
        ),
        "wide_edge_size_mb": file_size_mb(
            paths["edge"]
        ),
        "long_edge_size_mb": file_size_mb(
            paths["long"]
        ),
        "node_intensity_size_mb": file_size_mb(
            paths["node_intensity"]
        ),
        "wide_edge_rows": len(edges),
        "long_layer_edge_rows": len(long_edges),
        "node_intensity_rows": len(node_intensity),
    }

    atomic_write_json(
        success_payload,
        paths["success"],
    )

    if paths["failure"].exists():
        paths["failure"].unlink()

    if (
        CLEAN_DAILY_SUMMARIES_AFTER_SUCCESS
        and paths["daily_dir"].exists()
    ):
        shutil.rmtree(paths["daily_dir"])

    if paths["temp"].exists():
        shutil.rmtree(paths["temp"])

    if (
        not KEEP_WORKWEEK_OD
        and paths["workweek_od"].exists()
    ):
        paths["workweek_od"].unlink()

    if (
        not KEEP_RECURRENT_PARQUET
        and paths["recurrent"].exists()
    ):
        paths["recurrent"].unlink()

    print(
        f"[{month}] SUCCESS — "
        f"{len(edges):,} wide edges; "
        f"{len(long_edges):,} positive layer edges; "
        f"{month_elapsed / 60.0:.2f} min."
    )

    return {
        "month": month,
        "run_id": RUN_ID,
        "status": "success",
        "message": "completed",
        "elapsed_minutes": month_elapsed / 60.0,
        "config_signature": CONFIG_SIGNATURE,
        "updated_at": datetime.now().isoformat(
            timespec="seconds"
        ),
    }

## 10. Run the batch pipeline

In [ ]:
run_records = []
months_attempted = 0

if RUN_PIPELINE:
    for plan_row in processing_plan.itertuples(index=False):
        month = plan_row.month

        complete, _ = month_is_complete(month)
        force = month in FORCE_REPROCESS_MONTHS
        unfinished = not complete or force
        if MAX_MONTHS_PER_RUN is not None and unfinished and months_attempted >= MAX_MONTHS_PER_RUN:
            print("Reached MAX_MONTHS_PER_RUN; remaining months are left for the next run.")
            break
        if unfinished:
            months_attempted += 1

        print("\n" + "=" * 78)
        print("MONTH:", month)
        print("=" * 78)

        try:
            record = process_month(month, plan_row._asdict())
        except Exception as error:
            paths = month_paths(month)
            error_payload = {
                "month": month,
                "status": "failed",
                "config_signature": CONFIG_SIGNATURE,
                "run_id": RUN_ID,
                "failed_at": datetime.now().isoformat(timespec="seconds"),
                "error_type": type(error).__name__,
                "error": str(error),
                "traceback": traceback.format_exc(),
            }
            atomic_write_json(error_payload, paths["failure"])
            record = {
                "month": month,
                "run_id": RUN_ID,
                "status": "failed",
                "message": f"{type(error).__name__}: {error}",
                "elapsed_minutes": np.nan,
                "config_signature": CONFIG_SIGNATURE,
                "updated_at": datetime.now().isoformat(timespec="seconds"),
            }
            print(f"[{month}] FAILED: {type(error).__name__}: {error}")
            print("Partial outputs were retained for the next run.")
            if STOP_ON_ERROR:
                run_records.append(record)
                upsert_csv(record, QC_DIR / "monthly_processing_status.csv", keys=["month"])
                raise

        run_records.append(record)
        upsert_csv(record, QC_DIR / "monthly_processing_status.csv", keys=["month"])
else:
    print("RUN_PIPELINE=False. The plan was created but no month was processed.")

run_results = pd.DataFrame(run_records)
if not run_results.empty:
    atomic_write_csv(run_results, RUN_LOG_DIR / f"run_month_results_{RUN_ID}.csv")
    display(run_results)

## 11. Run summary and checkpoint status

In [ ]:
run_elapsed = time.perf_counter() - RUN_PERF_START

status_records = []
for month in all_months:
    complete, reason = month_is_complete(month)
    paths = month_paths(month)
    failure = safe_read_json(paths["failure"])
    status_records.append({
        "month": month,
        "complete": complete,
        "completion_check": reason,
        "wide_edge_exists": paths["edge"].exists(),
        "long_edge_exists": paths["long"].exists(),
        "node_intensity_exists": paths["node_intensity"].exists(),
        "workweek_od_exists": paths["workweek_od"].exists(),
        "recurrent_exists": paths["recurrent"].exists(),
        "partial_daily_files": (
            len(list(paths["daily_dir"].glob("*.parquet")))
            if paths["daily_dir"].exists() else 0
        ),
        "last_failure": failure.get("error") if failure else None,
    })

resume_status = pd.DataFrame(status_records)
atomic_write_csv(resume_status, QC_DIR / "resume_status.csv")

run_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_started_at": RUN_STARTED_AT.isoformat(timespec="seconds"),
    "run_finished_at": datetime.now().isoformat(timespec="seconds"),
    "total_elapsed_minutes": run_elapsed / 60,
    "total_elapsed_hours": run_elapsed / 3600,
    "months_requested": len(all_months),
    "months_complete": int(resume_status["complete"].sum()),
    "months_incomplete": int((~resume_status["complete"]).sum()),
    "layer_network_storage_mb": directory_size_mb(LAYER_NETWORK_DIR),
    "intermediate_storage_mb": directory_size_mb(PROJECT_ROOT / "01_intermediate"),
    "temporary_storage_mb": directory_size_mb(TEMP_DIR),
    "config_signature": CONFIG_SIGNATURE,
}])
atomic_write_csv(run_summary, RUN_LOG_DIR / f"run_summary_{RUN_ID}.csv")

print("\n========== RUN FINISHED ==========")
print("Run ID:", RUN_ID)
print("Elapsed:", f"{run_elapsed/60:.2f} minutes")
print("Completed months:", int(resume_status["complete"].sum()), "/", len(all_months))
print("Resume status:", QC_DIR / "resume_status.csv")
print("Monthly status:", QC_DIR / "monthly_processing_status.csv")

display(run_summary.T)
display(resume_status)

## 12. Layer metadata

In [ ]:
layer_metadata = pd.DataFrame([
    {
        "recurrence_band": band,
        "recurrence_midpoint": RECURRENCE_MIDPOINT[band],
        "recurrence_share_14d": RECURRENCE_MIDPOINT[band] / 14.0,
        "hybrid_intensity_score": HYBRID_INTENSITY_SCORE[band],
        "hybrid_rank_score": HYBRID_RANK_SCORE[band],
        "weight_column": LAYER_WEIGHT_COLUMNS[band],
        "layer_label": RECURRENCE_LAYER_LABEL[band],
        "interpretation_note": (
            "Official MITMA recurrence band over fourteen natural days; "
            "not a direct count of remote-working days."
        ),
    }
    for band in RECURRENCE_BAND_ORDER
])

atomic_write_csv(
    layer_metadata,
    LAYER_NETWORK_DIR / "recurrence_layer_metadata.csv",
)

display(layer_metadata)

## 13. Rerunning and rebuilding selected months

For a standard resumed run, retain:

```python
SKIP_COMPLETED_MONTHS = True
REUSE_PARTIAL_OUTPUTS = True
```

The pipeline skips months with a valid `_SUCCESS.json` marker and resumes
months that failed or remain incomplete.

To rebuild only selected months:

```python
FORCE_REPROCESS_MONTHS = {"2023-05", "2024-02"}
```

To process at most five unfinished months in one run:

```python
MAX_MONTHS_PER_RUN = 5
```

Daily Parquet checkpoints, intermediate recurrence tables, and workweek OD
tables are retained for failed months. After resolving a download or schema
issue, rerun the notebook to continue from the available checkpoints.
